In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
from scipy import stats
import utils
import seaborn as sns
import pingouin as pg
import matplotlib.cm as cm

import statsmodels.stats.power as smp
from statsmodels.stats.anova import AnovaRM
from tqdm import tqdm


from natsort import index_natsorted

# plt.rcParams['font.family'] = 'Times New Roman'
# plt.rcParams['font.family'] = 'Calibri'

path_figs = "./Figs/"

fingers = ['1', '2', '3', '4', '5']

iti = 1000 # msecs for inter-trial interval
planTime = 2000 # msecs for precue time
feedbackTime = 2000 # msecs for feedback time


total_sub_num = 10
num_sessions = 4
num_blocks_first_session = 4
num_blocks_second_session = 4
num_blocks_third_session = 4
num_blocks_fourth_session = 4
num_trials_per_block = 120
num_trials_baseline = 20

sub_nums = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

utils.set_figure_style("1col")
sns.color_palette('colorblind')


[(0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
 (0.8705882352941177, 0.5607843137254902, 0.0196078431372549),
 (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
 (0.8352941176470589, 0.3686274509803922, 0.0),
 (0.8, 0.47058823529411764, 0.7372549019607844),
 (0.792156862745098, 0.5686274509803921, 0.3803921568627451),
 (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
 (0.5803921568627451, 0.5803921568627451, 0.5803921568627451),
 (0.9254901960784314, 0.8823529411764706, 0.2),
 (0.33725490196078434, 0.7058823529411765, 0.9137254901960784)]

In [2]:
# reload utils
import importlib
importlib.reload(utils)

<module 'utils' from '/Users/amin/projects/LearningDynamics/ChordAdaptationDynamics/Version3/utils.py'>

In [3]:
subjs_list = utils.read_dat_files_subjs_list(sub_nums)

subjs = pd.concat(subjs_list, ignore_index=True)

subjs['TotalTrialNum'] = ((subjs['BN']-1)//2) * (num_trials_per_block + num_trials_baseline) + (subjs['BN']-1)%2 * num_trials_baseline + subjs['TN']
session_edges = np.cumsum([num_blocks_first_session, num_blocks_second_session, num_blocks_third_session, num_blocks_fourth_session])
subjs['day'] = np.searchsorted(session_edges, subjs['BN'], side = 'left') + 1
subjs['chord'] = subjs.apply(utils.extract_chord_form_target_force, axis=1)
subjs['block_type'] = subjs.apply(utils.extract_block_type, axis=1)
subjs['trial_num_within_chord'] = subjs.apply(utils.extract_trial_num_within_chord, axis=1)
subjs['count'] = 1


subjs['num_targets'] = subjs[[f'targetForce{i}' for i in range(1, 6)]].ne(0).sum(axis=1)
subjs.reset_index(drop=True, inplace=True)



In [4]:
subjs.columns

Index(['BN', 'TN', 'subNum', 'targetForce1', 'targetForce2', 'targetForce3',
       'targetForce4', 'targetForce5', 'endForce1', 'endForce2', 'endForce3',
       'endForce4', 'endForce5', 'endForcePurturbed1', 'endForcePurturbed2',
       'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
       'isTargetVisible', 'purturbation1', 'purturbation2', 'planTime',
       'feedbackTime', 'iti', 'forceGain', 'trialCorr', 'trialErrorType',
       'TotalTrialNum', 'day', 'chord', 'block_type', 'trial_num_within_chord',
       'count', 'num_targets'],
      dtype='str')

In [5]:
subjs = subjs[['subNum', 'BN', 'TN', 'trial_num_within_chord', 'TotalTrialNum', 'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5',
            'endForce1', 'endForce2', 'endForce3', 'endForce4', 'endForce5', 'isTargetVisible',
            'endForcePurturbed1', 'endForcePurturbed2', 'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
            'purturbation1', 'purturbation2', 'forceGain','trialCorr', 'trialErrorType', 'num_targets',
            'chord', 'day', 'block_type', 'count']]

In [6]:
subjs.head()

,subNum,BN,TN,trial_num_within_chord,TotalTrialNum,targetForce1,targetForce2,targetForce3,targetForce4,targetForce5,...,purturbation1,purturbation2,forceGain,trialCorr,trialErrorType,num_targets,chord,day,block_type,count
0,2,1,1,1,1,0.0,-2.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0,1,1,-v---,1,unpurturbed,1
1,2,1,2,2,2,0.0,-2.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0,1,1,-v---,1,unpurturbed,1
2,2,1,3,3,3,0.0,-2.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0,2,1,-v---,1,unpurturbed,1
3,2,1,4,4,4,0.0,-2.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0,1,1,-v---,1,unpurturbed,1
4,2,1,5,5,5,0.0,-2.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1,0,1,-v---,1,unpurturbed,1


In [7]:
subjs

,subNum,BN,TN,trial_num_within_chord,TotalTrialNum,targetForce1,targetForce2,targetForce3,targetForce4,targetForce5,...,purturbation1,purturbation2,forceGain,trialCorr,trialErrorType,num_targets,chord,day,block_type,count
0,2,1,1,1,1,0.0,-2.0,0.0,0.0,0.0,...,0.00,0.00,1.0,0,1,1,-v---,1,unpurturbed,1
1,2,1,2,2,2,0.0,-2.0,0.0,0.0,0.0,...,0.00,0.00,1.0,0,1,1,-v---,1,unpurturbed,1
2,2,1,3,3,3,0.0,-2.0,0.0,0.0,0.0,...,0.00,0.00,1.0,0,2,1,-v---,1,unpurturbed,1
3,2,1,4,4,4,0.0,-2.0,0.0,0.0,0.0,...,0.00,0.00,1.0,0,1,1,-v---,1,unpurturbed,1
4,2,1,5,5,5,0.0,-2.0,0.0,0.0,0.0,...,0.00,0.00,1.0,1,0,1,-v---,1,unpurturbed,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13715,14,4,116,136,276,0.0,2.0,0.0,-2.0,0.0,...,0.17,0.29,1.0,1,0,2,-^-v-,1,purturbed,1
13716,14,4,117,137,277,0.0,2.0,0.0,-2.0,0.0,...,0.35,0.57,1.0,1,0,2,-^-v-,1,purturbed,1
13717,14,4,118,138,278,0.0,2.0,0.0,-2.0,0.0,...,-0.76,-0.22,1.0,0,1,2,-^-v-,1,purturbed,1
13718,14,4,119,139,279,0.0,2.0,0.0,-2.0,0.0,...,0.82,-0.55,1.0,0,1,2,-^-v-,1,purturbed,1


In [8]:
subjs[['endForce1', 'endForce2', 'endForce3', 'endForce4', 'endForce5', 
'endForcePurturbed1', 'endForcePurturbed2', 'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
'purturbation1', 'purturbation2', 'num_targets', 'block_type', 'chord', 'day']].head()

,endForce1,endForce2,endForce3,endForce4,endForce5,endForcePurturbed1,endForcePurturbed2,endForcePurturbed3,endForcePurturbed4,endForcePurturbed5,purturbation1,purturbation2,num_targets,block_type,chord,day
0,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,0.0,0.0,1,unpurturbed,-v---,1
1,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,0.0,0.0,1,unpurturbed,-v---,1
2,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,0.0,0.0,1,unpurturbed,-v---,1
3,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,-6.277439e+66,0.0,0.0,1,unpurturbed,-v---,1
4,-1.090000e-01,-1.372000e+00,-1.020000e-01,1.600000e-02,-1.000000e-03,-1.090000e-01,-1.372000e+00,-1.020000e-01,1.600000e-02,-1.000000e-03,0.0,0.0,1,unpurturbed,-v---,1


In [9]:
subjs.to_csv(utils.path_misc+'subjs.csv', index = False, sep = '\t')